# Fraud Detection — Exploratory Data Analysis

**Goal of this notebook**

Document the dataset we train on. We answer five questions a Risk team will always ask before approving a new model:

1. How prevalent is fraud? (class imbalance)
2. What does a legitimate transaction look like vs. a fraudulent one?
3. Which features carry signal? (rough univariate analysis)
4. Are the fraud patterns we want to catch present in the data?
5. Do training and test splits look statistically comparable?

**How to run**

```bash
make data       # generates the parquet files used below
jupyter lab notebooks/01_eda.ipynb
```

If you have not generated the data yet, the first cell does it inline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fraud_detection.data.synthetic import generate_transactions, save_dataset
from fraud_detection.features.build_features import build_features

DATA_DIR = Path("..") / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DATA_DIR / "transactions_train.parquet"
TEST_PATH = DATA_DIR / "transactions_test.parquet"

if not TRAIN_PATH.exists():
    print("Generating dataset (one-off, ~30s)...")
    df = generate_transactions(n_rows=80_000, fraud_rate=0.015, n_customers=8_000, seed=42)
    save_dataset(df, DATA_DIR)

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")

## 1. Class imbalance

Fraud rate is around 1-2% in real card portfolios. We tune our metrics around this regime (PR AUC, recall at low FPR) rather than ROC AUC alone.

In [ ]:
summary = pd.DataFrame(
    {
        "train": [len(train_df), train_df["is_fraud"].sum(), train_df["is_fraud"].mean()],
        "test":  [len(test_df), test_df["is_fraud"].sum(), test_df["is_fraud"].mean()],
    },
    index=["n_rows", "n_fraud", "fraud_rate"],
)
summary

## 2. Amount distribution: legit vs fraud

We work on the log scale because amounts are very right-skewed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [(0, "#4C72B0"), (1, "#C44E52")]:
    sample = train_df.loc[train_df["is_fraud"] == label, "amount"]
    ax.hist(
        np.log1p(sample),
        bins=60,
        alpha=0.55,
        label="fraud" if label else "legit",
        color=color,
        density=True,
    )
ax.set_xlabel("log1p(amount EUR)")
ax.set_ylabel("density")
ax.set_title("Amount distribution (log scale) by class")
ax.legend()
plt.tight_layout()

## 3. Are the five fraud patterns present?

The synthetic generator produces five attack patterns. We check that all of them are represented in the fraud subset of the training data.

In [ ]:
fraud_only = train_df[train_df["is_fraud"] == 1]
fraud_only["fraud_pattern"].value_counts(normalize=True).rename("share")

## 4. Engineered features: signal check

Reproducing the feature engineering used at training and inference time, then comparing means by class.

In [ ]:
features = build_features(train_df)
FEATURES_OF_INTEREST = [
    "amount_log",
    "amount_zscore_30d",
    "amount_ratio_avg_30d",
    "velocity_score",
    "is_burst_1h",
    "is_geo_mismatch",
    "is_cnp",
    "is_ecom",
    "is_atm",
    "is_high_risk_mcc",
    "is_night",
    "is_weekend",
    "distinct_countries_last_24h",
    "n_tx_last_1h",
    "n_tx_last_24h",
]
means_by_class = features.groupby("is_fraud")[FEATURES_OF_INTEREST].mean().T
means_by_class.columns = ["legit", "fraud"]
means_by_class["lift_fraud_over_legit"] = (
    means_by_class["fraud"] / means_by_class["legit"].replace(0, np.nan)
).round(2)
means_by_class.sort_values("lift_fraud_over_legit", ascending=False)

## 5. Train / test consistency

Time-based split: oldest 80% to train, newest 20% to test. We confirm fraud rate and a few key features stay in the same ballpark.

In [ ]:
train_feats = build_features(train_df)
test_feats = build_features(test_df)
checks = pd.DataFrame(
    {
        "train_mean": train_feats[FEATURES_OF_INTEREST].mean(),
        "test_mean":  test_feats[FEATURES_OF_INTEREST].mean(),
    }
)
checks["rel_diff_%"] = ((checks["test_mean"] - checks["train_mean"]) / checks["train_mean"] * 100).round(2)
checks

## 6. Time of day

We expect fraud to be over-represented at night (0-5 AM).

In [ ]:
tmp = train_feats.groupby(["hour", "is_fraud"]).size().unstack(fill_value=0)
tmp_norm = tmp.div(tmp.sum(axis=0), axis=1)
ax = tmp_norm.plot(figsize=(10, 4))
ax.set_ylabel("share of class")
ax.set_title("Hourly distribution of transactions, legit vs fraud")
ax.legend(["legit", "fraud"])
plt.tight_layout()

## 7. Geo mismatch

Card country != merchant country is one of the strongest signals across portfolios.

In [ ]:
geo = train_feats.groupby(["is_geo_mismatch", "is_fraud"]).size().unstack(fill_value=0)
geo["fraud_rate"] = (geo[1] / (geo[0] + geo[1])).round(4)
geo

## 8. Correlations

Sanity check on multicollinearity. Tree models handle it fine, but we want to know it's there.

In [ ]:
corr = train_feats[FEATURES_OF_INTEREST + ["is_fraud"]].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation matrix")
plt.tight_layout()

## Takeaways

- Class imbalance is mild (~1.5%) but real; use PR AUC and recall @ low FPR.
- `is_geo_mismatch`, `amount_zscore_30d`, `n_tx_last_1h` show the strongest separation by class. This matches what the LightGBM importance ranks at the end of training.
- Fraud is over-represented at night, as expected.
- The time-based split is stable: feature means differ by less than a few percent between train and test.